## Cellule 1 : Imports et chargement des données nettoyées

In [2]:
import sqlite3
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)

# Load the cleaned dataset produced in notebook 01
df = pd.read_csv("...data/marketing_campaign_clean.csv")

# Detect the enrolment date column (DtCustomer or Dt_Customer) and standardize its name
date_col = next(c for c in df.columns if c.lower().replace("_", "") == "dtcustomer")
df = df.rename(columns={date_col: "DtCustomer"})

# Dates are strings again after the CSV round trip: parse them, then store them
# as ISO text (YYYY-MM-DD) so SQLite date functions like strftime() work
df["DtCustomer"] = pd.to_datetime(df["DtCustomer"], errors="coerce")
df["DtCustomer"] = df["DtCustomer"].dt.strftime("%Y-%m-%d")

print(df.shape)
df.head()

(2240, 30)


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,DtCustomer,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response,Income_outlier
0,5524,1957,Graduate,Single,58138.0,0.0,0.0,2012-09-04,58.0,635.0,88.0,546.0,172.0,88.0,88.0,3.0,8.0,10.0,4.0,7.0,0,0,0,0,0,0,3,11,1,0
1,2174,1954,Graduate,Single,46344.0,1.0,1.0,2014-03-08,38.0,11.0,1.0,6.0,2.0,1.0,6.0,2.0,1.0,1.0,2.0,5.0,0,0,0,0,0,0,3,11,0,0
2,4141,1965,Graduate,Together,71613.0,0.0,0.0,2013-08-21,26.0,426.0,49.0,127.0,111.0,21.0,42.0,1.0,8.0,2.0,10.0,4.0,0,0,0,0,0,0,3,11,0,0
3,6182,1984,Graduate,Together,26646.0,1.0,0.0,2014-02-10,26.0,11.0,4.0,20.0,10.0,3.0,5.0,2.0,2.0,0.0,4.0,6.0,0,0,0,0,0,0,3,11,0,0
4,5324,1981,PhD,Married,58293.0,1.0,0.0,2014-01-19,94.0,173.0,43.0,118.0,46.0,27.0,15.0,5.0,5.0,3.0,6.0,5.0,0,0,0,0,0,0,3,11,0,0


## Cellule 2 : Connexion SQLite et création de la table

In [9]:
# Connect to a local SQLite database ()
conn = sqlite3.connect("analysis.db")

# Push the dataframe into a SQL table called 'sales'
df.to_sql("sales", conn, if_exists="replace", index=False)

# Sanity check: row count and table structure
print(pd.read_sql("SELECT COUNT(*) AS nb_rows FROM sales", conn))
print(pd.read_sql("PRAGMA table_info(sales)", conn)[["name", "type"]])

   nb_rows
0     2240
                   name     type
0                    ID  INTEGER
1            Year_Birth  INTEGER
2             Education     TEXT
3        Marital_Status     TEXT
4                Income     REAL
5               Kidhome     REAL
6              Teenhome     REAL
7            DtCustomer     TEXT
8               Recency     REAL
9              MntWines     REAL
10            MntFruits     REAL
11      MntMeatProducts     REAL
12      MntFishProducts     REAL
13     MntSweetProducts     REAL
14         MntGoldProds     REAL
15    NumDealsPurchases     REAL
16      NumWebPurchases     REAL
17  NumCatalogPurchases     REAL
18    NumStorePurchases     REAL
19    NumWebVisitsMonth     REAL
20         AcceptedCmp3  INTEGER
21         AcceptedCmp4  INTEGER
22         AcceptedCmp5  INTEGER
23         AcceptedCmp1  INTEGER
24         AcceptedCmp2  INTEGER
25             Complain  INTEGER
26        Z_CostContact  INTEGER
27            Z_Revenue  INTEGER
28             Respon

## Cellule 3 : Requête 1, taux d’acceptation par niveau d’éducation

In [14]:
# Q1: Response rate and average spending by education level
query_1 = """
SELECT
    Education,
    COUNT(*)                                        AS nb_customers,
    SUM(Response)                                   AS nb_accepted,
    ROUND(100.0 * SUM(Response) / COUNT(*), 2)      AS response_rate_pct,
    ROUND(AVG(MntWines + MntFruits + MntMeatProducts + MntFishProducts
              + MntSweetProducts + MntGoldProds), 2) AS avg_total_spent
FROM sales
GROUP BY Education
ORDER BY response_rate_pct DESC;
"""
q1 = pd.read_sql(query_1, conn)
q1

,Education,nb_customers,nb_accepted,response_rate_pct,avg_total_spent
0,PhD,486,101,20.78,672.41
1,Master,573,79,13.79,570.95
2,Graduate,1127,152,13.49,619.90
3,Basic,54,2,3.70,81.80


## Cellule 4 : Requête 2, segmentation par quartile de revenu

In [19]:
# Q2: Customer segments by income quartile (1 = lowest income, 4 = highest)
query_2 = """
WITH spend AS (
    SELECT
        Income,
        Response,
        NumWebPurchases + NumCatalogPurchases + NumStorePurchases AS nb_purchases,
        MntWines + MntFruits + MntMeatProducts + MntFishProducts
            + MntSweetProducts + MntGoldProds                     AS total_spent,
        NTILE(4) OVER (ORDER BY Income)                           AS income_quartile
    FROM sales
    WHERE Income IS NOT NULL
)
SELECT
    income_quartile,
    COUNT(*)                                    AS nb_customers,
    ROUND(MIN(Income), 0)                       AS min_income,
    ROUND(MAX(Income), 0)                       AS max_income,
    ROUND(AVG(total_spent), 2)                  AS avg_total_spent,
    ROUND(AVG(total_spent * 1.0 / NULLIF(nb_purchases, 0)), 2) AS avg_spent_per_purchase,
    ROUND(100.0 * SUM(Response) / COUNT(*), 2)  AS response_rate_pct
FROM spend
GROUP BY income_quartile
ORDER BY income_quartile;
"""
q2 = pd.read_sql(query_2, conn)
q2

,income_quartile,nb_customers,min_income,max_income,avg_total_spent,avg_spent_per_purchase,response_rate_pct
0,1,554,1730.0,35246.0,81.59,13.11,10.47
1,2,554,35322.0,51373.0,231.32,21.13,12.45
2,3,554,51390.0,68487.0,757.74,43.19,10.29
3,4,554,68627.0,666666.0,1357.65,72.75,26.90


## Cellule 5 : Requête 3, dépenses par catégorie de produit

In [24]:
# Q3: Total revenue and share of revenue by product category
query_3 = """
WITH categories AS (
    SELECT 'Wines'  AS category, SUM(MntWines)         AS total_spent FROM sales
    UNION ALL
    SELECT 'Meat',               SUM(MntMeatProducts)              FROM sales
    UNION ALL
    SELECT 'Gold',               SUM(MntGoldProds)                 FROM sales
    UNION ALL
    SELECT 'Fish',               SUM(MntFishProducts)              FROM sales
    UNION ALL
    SELECT 'Sweets',             SUM(MntSweetProducts)             FROM sales
    UNION ALL
    SELECT 'Fruits',             SUM(MntFruits)                    FROM sales
)
SELECT
    category,
    total_spent,
    ROUND(100.0 * total_spent / SUM(total_spent) OVER (), 2) AS share_pct
FROM categories
ORDER BY total_spent DESC;
"""
q3 = pd.read_sql(query_3, conn)
q3

,category,total_spent,share_pct
0,Wines,680816.0,50.17
1,Meat,373968.0,27.56
2,Gold,98609.0,7.27
3,Fish,84057.0,6.19
4,Sweets,60621.0,4.47
5,Fruits,58917.0,4.34


## Cellule 6 : Requête 4, canaux d’achat selon le profil familial

In [30]:
# Q4: Purchase channel behavior by household profile (children / teenagers)
query_4 = """
SELECT
    CASE
        WHEN Kidhome = 0 AND Teenhome = 0 THEN '1. No children'
        WHEN Kidhome > 0 AND Teenhome = 0 THEN '2. Small children only'
        WHEN Kidhome = 0 AND Teenhome > 0 THEN '3. Teenagers only'
        ELSE                                    '4. Small children + teenagers'
    END                                         AS household_profile,
    COUNT(*)                                    AS nb_customers,
    ROUND(AVG(NumWebPurchases), 2)              AS avg_web_purchases,
    ROUND(AVG(NumCatalogPurchases), 2)          AS avg_catalog_purchases,
    ROUND(AVG(NumStorePurchases), 2)            AS avg_store_purchases,
    ROUND(AVG(NumDealsPurchases), 2)            AS avg_deals_purchases,
    ROUND(AVG(NumWebVisitsMonth), 2)            AS avg_web_visits_month
FROM sales
GROUP BY household_profile
ORDER BY household_profile;
"""
q4 = pd.read_sql(query_4, conn)
q4

,household_profile,nb_customers,avg_web_purchases,avg_catalog_purchases,avg_store_purchases,avg_deals_purchases,avg_web_visits_month
0,1. No children,638,4.39,4.76,7.26,1.14,3.48
1,2. Small children only,520,2.76,0.82,3.64,2.14,6.83
2,3. Teenagers only,655,5.54,3.18,7.17,2.73,5.21
3,4. Small children + teenagers,427,3.00,0.98,4.09,3.70,6.39


## Cellule 7 : Requête 5, performance de chaque campagne marketing

In [35]:
# Q5: Acceptance rate of each marketing campaign
query_5 = """
SELECT 'Campaign 1' AS campaign, SUM(AcceptedCmp1) AS nb_accepted,
       ROUND(100.0 * SUM(AcceptedCmp1) / COUNT(*), 2) AS acceptance_rate_pct FROM sales
UNION ALL
SELECT 'Campaign 2', SUM(AcceptedCmp2),
       ROUND(100.0 * SUM(AcceptedCmp2) / COUNT(*), 2) FROM sales
UNION ALL
SELECT 'Campaign 3', SUM(AcceptedCmp3),
       ROUND(100.0 * SUM(AcceptedCmp3) / COUNT(*), 2) FROM sales
UNION ALL
SELECT 'Campaign 4', SUM(AcceptedCmp4),
       ROUND(100.0 * SUM(AcceptedCmp4) / COUNT(*), 2) FROM sales
UNION ALL
SELECT 'Campaign 5', SUM(AcceptedCmp5),
       ROUND(100.0 * SUM(AcceptedCmp5) / COUNT(*), 2) FROM sales
UNION ALL
SELECT 'Last campaign (Response)', SUM(Response),
       ROUND(100.0 * SUM(Response) / COUNT(*), 2) FROM sales;
"""
q5 = pd.read_sql(query_5, conn)
q5

,campaign,nb_accepted,acceptance_rate_pct
0,Campaign 1,144,6.43
1,Campaign 2,30,1.34
2,Campaign 3,163,7.28
3,Campaign 4,167,7.46
4,Campaign 5,163,7.28
5,Last campaign (Response),334,14.91
